## 0. Task

Strict span-F1 over 12 types. A span counts only if **start, end, and label** all match.

The first draft was a word-BiLSTM that stopped at 400 words — late entities scored zero. The selected system is **efficient full fine-tune + linear-chain CRF**, two seeds averaged.


In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
SEARCH = [HERE, HERE.parent, Path("/kaggle/working")]
for base in (HERE, HERE.parent, HERE.parent.parent, HERE.parent.parent.parent):
    SEARCH.append(base)
    for phase in range(1, 8):
        SEARCH.append(base / f"phase{phase}")
        SEARCH.append(base / f"phase{phase}" / "code")
        SEARCH.append(base / "code" / f"phase{phase}")
for folder in SEARCH:
    if folder.is_dir() and str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
print("python path heads:")
for folder in sys.path[:8]:
    print(" ", folder)


## 1. Labels


In [ ]:
LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO",
]
print(len(LABELS), "entity types")


## 2. Why character offsets on BPE tokens

The official scorer matches character intervals on the original UTF-8 string. Each BPE token stores `[start, end)`. A BIO tag is mapped back to that interval, never to a whitespace-normalised copy.


In [ ]:
print("See phase2_lner.py: sliding 1024-token windows, overlap merged by score.")
print("Causal mask is turned off for tagging (enable_full_attention) so a name can see the words after it.")


## 3. LoRMA vs full FT

LoRMA+ is compulsory on at least one architecture:

`h = (I + (α/r) B A) W x` with `B = 0` at init, so step 0 equals the frozen Phase-1 net.

NER still preferred **full FT** of the efficient body. LoRMA was the required PEFT zip.


In [ ]:
import inspect
from phase2_lner import LoRMALinear, LoRALinear, ChainCRF, enable_full_attention

print(inspect.getsource(LoRMALinear))
print("--- LoRA (additive) for comparison ---")
print(inspect.getsource(LoRALinear.forward))


## 4. Linear-chain CRF

Illegal BIO transitions (`I-DATE` after `B-COURT`) are banned in the transition matrix. Decode with Viterbi.


In [ ]:
print(inspect.getsource(ChainCRF.__init__))
print("decode = Viterbi; nll = logZ − gold path")


## 5. Selected serving path


In [ ]:
print("arch     : efficient")
print("method   : full fine-tune + CRF")
print("ensemble : seeds 42 and 7, average logits")
print("script   : phase2_lner.py")
print("command  : python phase2_lner.py --arch efficient --method full")


## 6. Zip


In [ ]:
print("prediction.json: {doc_id: [[start, end, LABEL], ...]}")
print("best zip: efficient_s42_s7_ensemble")
